# Case Study: Census

In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

## U.S. National Park Service

Source: [Responsible Datasets in Context](https://www.responsible-datasets-in-context.com/posts/np-data/)

<img src="https://www.nps.gov/common/uploads/structured_data/3C84CF74-1DD8-B71B-0B9C7FF83F7C68EB.jpg?width=990&height=742&mode=crop&quality=78" alt="photo of mountains and trees in Yosemite National Park" width="400 px" />

In [ ]:
# load in data
# recreational visits to U.S. national parks in 2024
nps = Table.read_table('US-National-Parks_RecreationVisits_1979-2024.csv')
nps = nps.where("Year", 2024).drop("Year", "Region").sort("Visits", 2024)
nps.show(3)

In [ ]:
# `take`
# 1. what is the tenth most visited park?
# 2. how do I see just that row?
# 3. how do I see what is in the row after that?

nps.take(9, 10)

In [ ]:
# more common: we will use `where` to pick out rows based on some condition
nps.where('State', 'CA').show(2)

In [ ]:
# ... or equivalently
nps.where('State', are.equal_to('CA')).show(2)

In [ ]:
# what about not equal to?
nps.where('State', are.not_equal_to('CA')).show(2)

In [ ]:
# more common: conditions with numerical values
nps.where('Visits', are.between(1e5, 1e6)).show(2)

In [ ]:
# what fraction of parks satisfy the above condition?
nps.where('Visits', are.between(1e5, 1e6)).num_rows / nps.num_rows

### Discussion Questions

The table `nps` has columns:


| Park | Park Code | State | Visits |
| :-- | :-- | :-- | :-- |
| ... | ... | ... | ... |


#### a) Create an **array** containing the names of all California parks that had more than than 1 million visitors.

In [ ]:
# ... code here ...


#### b) After evaluating these two expressions in order, what's the result of the second one?

In [ ]:
nps.drop('State')
nps.num_columns

<br/><br/><br/>

---

## Census ##

In [ ]:
full = Table.read_table('nc-est2025-agesex-res.csv')
full.show(5)

In [ ]:
partial = full.select('SEX', 'AGE', 'POPESTIMATE2020', 'POPESTIMATE2025')
partial.show(6)

### Newborns

**DISCUSS**: How can there be different values for newborn babies vs. five year olds five years later?

In [ ]:
partial.column(3).take(5) / partial.column(2).take(0)

In [ ]:
# make column labels a bit nicer
us_pop = partial.relabeled(2, '2020').relabeled(3, '2025')
us_pop.show(5)

### Elderly

In [ ]:
us_pop.where('AGE', are.above_or_equal_to(98)).sort('AGE').show()

In [ ]:
us_pop.where('SEX', 0).where('AGE', are.above_or_equal_to(98)).sort('AGE').show()

In [ ]:
# how would we see the percent change by each age group?
percent_change = (us_pop.column('2025') - us_pop.column('2020')) / us_pop.column('2020')
us_pop_change = us_pop.with_column("Percent Change", percent_change).set_format("Percent Change", PercentFormatter)
us_pop_change.where('SEX', 0).where('AGE', are.above_or_equal_to(98)).sort('AGE').show()

**DISCUSS**: Why are there so many more elderly in 2025?

### 2025 Sex Ratios

In [ ]:
# focus just on 2025 population estimates, now by male/female
us_pop_2025 = us_pop.drop('2020')
us_pop_2025.show(3)

In [ ]:
old = us_pop.where('AGE', 100)
old

In [ ]:
all_ages = us_pop_2025.where('AGE', are.equal_to(999))
all_ages

In [ ]:
infants = us_pop_2025.where('AGE', are.equal_to(0))
infants

In [ ]:
females_all_rows = us_pop_2025.where('SEX', are.equal_to(2))
females = females_all_rows.where('AGE', are.not_equal_to(999))
females.show(3)

In [ ]:
males_all_rows = us_pop_2025.where('SEX', are.equal_to(1))
males = males_all_rows.where('AGE', are.not_equal_to(999))
males.show(3)

How does the ratio of females to males vary by age?

In [ ]:
f_to_m_ratios = females.column(2) / males.column(2)

ratios = Table().with_columns(
    'Age', females.column('AGE'),
    'F:M Ratio', f_to_m_ratios
)

ratios

In [ ]:
ratios.sort('Age', descending=True)

#### Line Plot

In [ ]:
ratios.plot('Age', 'F:M Ratio')